In [1]:
import sys
import os
sys.path.append(os.path.abspath('../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.ensembles import *

from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import make_scorer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import integrated_brier_score
from sksurv.metrics import concordance_index_ipcw
from sksurv.nonparametric import kaplan_meier_estimator
from sklearn.inspection import permutation_importance

from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

import time
set_config(display="text")  # displays text representation of estimators

In [2]:
dataset = pd.read_csv("../data/data_final.csv")

diagnosis_variables = []

with open("../data/diagnosis_final.txt", "r") as file:
    for line in file:
        diagnosis_variables.append(line.strip())

treatment_variables = []
with open("../data/treatment_final.txt", "r") as file:
    for line in file:
        treatment_variables.append(line.strip())

survival_dataset = dataset.drop('Unnamed: 0', axis=1)

feature_columns = treatment_variables + diagnosis_variables

In [3]:
# Find duplicated patients
mask = (survival_dataset.duplicated(subset='PERSON_ID', keep=False)) #& (survival_dataset['Regional LN recurrence'] == 1)

# For each duplicated PERSON_ID with recurrence, keep only the row with the lowest survival days
to_remove = (
    survival_dataset[mask]
    .sort_values(['PERSON_ID', 'Survival days'], ascending=[True, True])
    .groupby('PERSON_ID')
    .head(-1)
    .index
)

survival_dataset = survival_dataset.drop(to_remove)

In [4]:
# Prepare data
data_x = survival_dataset[feature_columns]
data_y = Surv.from_arrays(
    event=survival_dataset['Regional LN recurrence'],
    time=survival_dataset['Survival days']
)

In [5]:
sgkf = StratifiedKFold(n_splits=5, shuffle=True, random_state=12)
cv = sgkf.split(X=data_x, y=survival_dataset['Regional LN recurrence'])

train_index, test_index = next(cv)
x_train = data_x.iloc[train_index]
y_train = data_y[train_index]

x_test = data_x.iloc[test_index]
y_test = data_y[test_index]

subsampling = survival_dataset.iloc[train_index].copy()

In [6]:
for train_idx, val_idx in cv:
    print("Train events:", survival_dataset.iloc[train_idx]["Regional LN recurrence"].sum())
    print("Val events:", survival_dataset.iloc[val_idx]["Regional LN recurrence"].sum())

Train events: 27.0
Val events: 6.0
Train events: 26.0
Val events: 7.0
Train events: 26.0
Val events: 7.0
Train events: 26.0
Val events: 7.0


In [7]:
param_grid = {
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 500],
    'max_depth': [1, 2, 4],
    'min_samples_split': [2, 5, 10],
    'n_controls_per_case': [1, 2, 4, 10, 15]
}

sgkf_1 = StratifiedKFold(n_splits=4, shuffle=True, random_state=12)
cv = sgkf_1.split(X=x_train, y=y_train['event'])

In [8]:
results_df, best_params, summary = generic_grid_search(
    GradientBoostingSurvivalAnalysis,
    x_train,
    y_train,
    cv,
    param_grid,
    metrics=['c-index', 'c-index-ipcw', 'int-brier-score'],
    model_name='GBM',
    scale_data=False,
    scaler_type='standard',
    verbose=True,
    random_state=12,
    n_jobs=-1,
    dataset=subsampling,
    feature_columns=feature_columns
)


GBM GRID SEARCH (Manual Implementation)
Model: GradientBoostingSurvivalAnalysis
Scaling: No
Subsampling: No
Metrics: ['c-index', 'c-index-ipcw', 'int-brier-score']
Parameter Grid:
  - learning_rate: [0.05, 0.1, 0.2]
  - n_estimators: [100, 200, 500]
  - max_depth: [1, 2, 4]
  - min_samples_split: [2, 5, 10]
  - n_controls_per_case: [1, 2, 4, 10, 15]
Total combinations: 405

[1/405] Testing params: {'learning_rate': 0.05, 'n_estimators': 100, 'max_depth': 1, 'min_samples_split': 2}, n_controls_per_case=1
  c-index: 0.8391 ± 0.0499
  c-index-ipcw: 0.7469 ± 0.0620
  int-brier-score: 0.0534 ± 0.0038
[2/405] Testing params: {'learning_rate': 0.05, 'n_estimators': 100, 'max_depth': 1, 'min_samples_split': 2}, n_controls_per_case=2
  Elapsed: 0.0min, Est. remaining: 16.2min
  c-index: 0.7916 ± 0.0572
  c-index-ipcw: 0.7222 ± 0.0245
  int-brier-score: 0.0341 ± 0.0058
[3/405] Testing params: {'learning_rate': 0.05, 'n_estimators': 100, 'max_depth': 1, 'min_samples_split': 2}, n_controls_per_ca

In [10]:
results_df.to_csv("../results/subsampling/gbm_subsampling_grid_search_results.csv", index=False)

Running the test on the best and correct parameters

In [77]:
# Load best hyperparameters from corrected_sub_results
best_results = pd.read_csv("../results/corrected_sub_results/gbm_subsampling_grid_search_results.csv")
# Select row with highest c-index-ipcw_mean
best_params = best_results.iloc[0] #.sort_values('c-index-ipcw_mean', ascending=False).iloc[0]

print("Best hyperparameters:")
print(f"  learning_rate: {best_params['learning_rate']}")
print(f"  n_estimators: {int(best_params['n_estimators'])}")
print(f"  max_depth: {int(best_params['max_depth'])}")
print(f"  min_samples_split: {int(best_params['min_samples_split'])}")
print(f"  n_controls_per_case: {int(best_params['n_controls_per_case'])}")
print(f"  c-index-ipcw_mean (CV): {best_params['c-index-ipcw_mean']:.4f}")
print()

Best hyperparameters:
  learning_rate: 0.1
  n_estimators: 100
  max_depth: 2
  min_samples_split: 2
  n_controls_per_case: 10
  c-index-ipcw_mean (CV): 0.7434



In [78]:
# Prepare training fold dataframe (subsampling variable was created earlier)
train_fold = subsampling.copy()

# Apply case-control subsampling
x_train_sub, y_train_sub, sub_dataset = sample_case_control(
    train_fold=train_fold,
    event_col='Regional LN recurrence',
    time_col='Survival days',
    n_controls_per_case=int(best_params['n_controls_per_case']),
    feature_columns=feature_columns,
    random_state=12
)

print(f"Original training samples: {len(x_train)}")
print(f"Subsampled training samples: {len(x_train_sub)}")
print(f"Test samples: {len(x_test)}")

Original training samples: 2376
Subsampled training samples: 291
Test samples: 594


In [79]:
# Train final model with best hyperparameters
final_model = GradientBoostingSurvivalAnalysis(
    learning_rate=best_params['learning_rate'],
    n_estimators=int(best_params['n_estimators']),
    max_depth=int(best_params['max_depth']),
    min_samples_split=int(best_params['min_samples_split']),
    random_state=12
)

final_model.fit(x_train_sub, y_train_sub)
print("Model trained successfully!")

Model trained successfully!


In [80]:
# Test the model on test set
# Calculate c-index
c_index = final_model.score(x_test, y_test)

# Calculate c-index IPCW
tau = y_train_sub["time"].max()
c_index_ipcw = concordance_index_ipcw(y_train_sub, y_test, final_model.predict(x_test), tau=tau)[0]

# Calculate integrated Brier score
surv_funcs = final_model.predict_survival_function(x_test)

# Use robust time filtering
max_time = min(y_train_sub['time'].max(), y_test['time'].max()) * 0.9

# Use percentile-based times (5% to 90%)
times = np.percentile(y_train_sub['time'], np.linspace(5, 90, 10))
# Ensure times are unique and sorted
times = np.unique(times)
# Filter times that are within valid range
times = times[(times > 0) & (times <= max_time)]

preds_at_times = np.array([[fn(t) for t in times] for fn in surv_funcs])
brier_score = integrated_brier_score(y_train_sub, y_test, preds_at_times, times)

print("\\nTest Set Performance:")
print(f"  C-index: {c_index:.4f}")
print(f"  C-index IPCW: {c_index_ipcw:.4f}")
print(f"  Integrated Brier Score: {brier_score:.4f}")

\nTest Set Performance:
  C-index: 0.8163
  C-index IPCW: 0.7982
  Integrated Brier Score: 0.0090


# Feature Importance for SHAP Analysis

In [81]:
'''# Extract feature importance from GBM model
# GBM provides feature_importances_ attribute
import pandas as pd
import numpy as np

# IMPORTANT: Use the actual feature names from the training data
# The model was trained on x_train_sub
actual_features = x_train_sub.columns.tolist()

# Get feature importances from the final GBM model
if len(final_model.feature_importances_) != len(actual_features):
    print(f"Warning: Importance length ({len(final_model.feature_importances_)}) doesn't match feature length ({len(actual_features)})")
    print(f"Using minimum length for mapping")
    min_len = min(len(final_model.feature_importances_), len(actual_features))
    importances = final_model.feature_importances_[:min_len]
    actual_features = actual_features[:min_len]
else:
    importances = final_model.feature_importances_

importance_df = pd.DataFrame({
    'Feature': actual_features,
    'Importance': importances
})

# Sort by importance (most important first)
importance_df = importance_df.sort_values('Importance', ascending=False)

# Display top 20 most important features
print("Top 20 Most Important Features for GBM Model:")
print("="*60)
print(importance_df.head(20).to_string(index=False))
print(f"\nTotal features: {len(importance_df)}")

# Store for SHAP analysis
top_features_gbm = importance_df.head(20)['Feature'].tolist()
print(f"\n\nTop features stored in 'top_features_gbm' variable")
print(f"All importances stored in 'importance_df' variable")'''

'# Extract feature importance from GBM model\n# GBM provides feature_importances_ attribute\nimport pandas as pd\nimport numpy as np\n\n# IMPORTANT: Use the actual feature names from the training data\n# The model was trained on x_train_sub\nactual_features = x_train_sub.columns.tolist()\n\n# Get feature importances from the final GBM model\nif len(final_model.feature_importances_) != len(actual_features):\n    print(f"Warning: Importance length ({len(final_model.feature_importances_)}) doesn\'t match feature length ({len(actual_features)})")\n    print(f"Using minimum length for mapping")\n    min_len = min(len(final_model.feature_importances_), len(actual_features))\n    importances = final_model.feature_importances_[:min_len]\n    actual_features = actual_features[:min_len]\nelse:\n    importances = final_model.feature_importances_\n\nimportance_df = pd.DataFrame({\n    \'Feature\': actual_features,\n    \'Importance\': importances\n})\n\n# Sort by importance (most important first)\

In [82]:
'''# Export feature importance to CSV
importance_df.to_csv("../results/features/gbm_feature_importance.csv", index=False)
print("Feature importance exported to: ../results/features/gbm_feature_importance.csv")'''

'# Export feature importance to CSV\nimportance_df.to_csv("../results/features/gbm_feature_importance.csv", index=False)\nprint("Feature importance exported to: ../results/features/gbm_feature_importance.csv")'